In [376]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.preprocessing import PolynomialFeatures


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor

from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [377]:
data_path = "../joined_features/full_dataset.csv"
df = pd.read_csv(data_path)
########## ---------- Convert into categorical ---------- ########## 
topic_cols = [
    'topic_activity_0', 'topic_activity_1', 'topic_activity_2',
    'topic_activity_3', 'topic_activity_4', 'topic_activity_5',
    'topic_activity_6', 'topic_activity_7', 'topic_activity_8',
    'topic_activity_9', 'topic_activity_10', 'topic_activity_11'
]

# find max column per row
max_cols = df[topic_cols].idxmax(axis=1)

# zero everything
df[topic_cols] = 0

# set the max column to 1
for col in topic_cols:
    df.loc[max_cols == col, col] = 1

# Remove rows with no posts (these can introduce NaNs)
df = df[df["post_count"] != 0]

# Basic NaN diagnostics
nan_count = df.isna().sum().sum()
print(f"Number of NaN values: {nan_count}")

if nan_count > 0:
    failing_cols = df.columns[df.isna().any()]
    print(f"Columns with NaN: {list(failing_cols)}")

Number of NaN values: 1
Columns with NaN: ['avg_text_len']


In [ ]:
features = [
    'topic_activity_0', 'topic_activity_1', 'topic_activity_2',
    'topic_activity_3', 'topic_activity_4', 'topic_activity_5',
    'topic_activity_6', 'topic_activity_7', 'topic_activity_8',
    'topic_activity_9', 'topic_activity_10', 'topic_activity_11',
    'topic_activity_12', 'trump_sentiment_mean_x', 'trump_sentiment_std',
    'trump_sentiment_pct_negative', 
    'cpi', 
    'interest_rate', 
    'consumer_sentiment',
    'unemployment'
]

predictors = [
    'news_sentiment_rolling', 'news_sentiment_mean_y'
]
df = df.sort_index()  # or: df = df.sort_values("date")
n = len(df)



train_end = int(n * 0.7)
val_end   = int(n * 0.85)

train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]

X_train = train[features]
Y_train = train[predictors]

X_val = val[features]
Y_val = val[predictors]

X_test = test[features]
Y_test = test[predictors]

In [379]:
models = {
    "random_forest": MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    ),

    "linear_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MultiOutputRegressor(LinearRegression()))
    ]),

    "svm": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MultiOutputRegressor(
            SVR(kernel="rbf", C=1.0)
        ))
    ]),

    "xgboost": MultiOutputRegressor(
        XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror"
        )
    ),

    "lightgbm": MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    ),

    "mlp": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=1000,
            random_state=42
        ))
    ])
}

In [380]:

def evaluate_model(name, model, X_train, Y_train, X_val, Y_val):
    model.fit(X_train, Y_train)

    preds = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(Y_val, preds))
    mae = mean_absolute_error(Y_val, preds)
    r2 = r2_score(Y_val, preds)

    return {
        "model_name": name,
        "model": model,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }


def run_all_models(models, X_train, Y_train, X_val, Y_val):
    results = []

    for name, model in models.items():
        print(f"Training {name}...")

        result = evaluate_model(
            name,
            model,
            X_train,
            Y_train,
            X_val,
            Y_val
        )

        results.append(result)

    results_df = pd.DataFrame(results).drop(columns=["model"])
    results_df = results_df.sort_values("rmse")

    return results, results_df

In [381]:
results, results_df = run_all_models(
    models,
    X_train,
    Y_train,
    X_val,
    Y_val
)

print(results_df)

Training random_forest...
Training linear_regression...
Training svm...
Training xgboost...
Training lightgbm...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000056 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 138
[LightGBM] [Info] Number of data points in the train set: 102, number of used features: 10
[LightGBM] [Info] Start training from score -0.051773
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

In [382]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results = df_results.drop(columns=["model"]) 
df_results = df_results.sort_values("rmse")

print(df_results)

          model_name      rmse       mae        r2
1  linear_regression  0.278152  0.240706 -2.743855
0      random_forest  0.288727  0.252802 -3.012795
3            xgboost  0.290897  0.252069 -3.072561
4           lightgbm  0.294487  0.256605 -3.171236
2                svm  0.305243  0.273651 -3.497115
5                mlp  0.319265  0.273154 -3.911130


In [383]:
baseline_preds = Y_test.shift(1)

print("Baseline R2:", r2_score(
    Y_test.iloc[1:], baseline_preds.iloc[1:]
))

Baseline R2: 0.019526550193576075
